In [75]:
xbuf = Buffer(x, axis=0).arr.T
z = np.zeros((3,1))
np.concat([xbuf, z], axis=1)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from IPython.display import display_svg
from typing import List, Tuple
from pathlib import Path
import random
import torch
import numpy as np
import pandas as pd
from itertools import combinations_with_replacement
from typing import Callable, Any
import pyrtl
from pyrtl.rtllib.libutils import twos_comp_repr, rev_twos_comp_repr
from pyrtl.rtllib.multipliers import generalized_fma
from pyrtl import (
    WireVector, 
    Const, 
    Input,
    Output, 
    Register, 
    conditional_assignment,
    otherwise,
    concat,
    Simulation, 
    SimulationTrace, 
    reset_working_block
)
from kai.src.utils import custom_render_trace, basic_circuit_analysis
from kai.src.bfloat16 import BF16

In [3]:
output_dir = Path('kai/output')
output_dir

In [4]:
E_BITS  = 8
M_BITS  = 7
MSB     = E_BITS + M_BITS

In [5]:
pyrtl.set_debug_mode(False)

### PyRTL Simple Pipeline


In [6]:
class SimplePipeline(object):
    """ Pipeline builder with auto generation of pipeline registers. """

    def __init__(self):
        self._pipeline_register_map = {}
        self._current_stage_num = 0
        stage_list = [method for method in dir(self) if method.startswith('stage')]
        for stage in sorted(stage_list):
            stage_method = getattr(self, stage)
            stage_method()
            self._current_stage_num += 1

    def __getattr__(self, name):
        try:
            return self._pipeline_register_map[self._current_stage_num][name]
        except KeyError:
            raise pyrtl.PyrtlError(
                'error, no pipeline register "%s" defined for stage %d'
                % (name, self._current_stage_num))

    def __setattr__(self, name, value):
        if name.startswith('_'):
            # do not do anything tricky with variables starting with '_'
            object.__setattr__(self, name, value)
        else:
            next_stage = self._current_stage_num + 1
            pipereg_id = str(self._current_stage_num) + 'to' + str(next_stage)
            rname = 'pipereg_' + pipereg_id + '_' + name
            new_pipereg = pyrtl.Register(bitwidth=len(value), name=rname)
            if next_stage not in self._pipeline_register_map:
                self._pipeline_register_map[next_stage] = {}
            self._pipeline_register_map[next_stage][name] = new_pipereg
            new_pipereg.next <<= value

## Some Basic MAC Examples


### Fully combinatorial


In [7]:
def combinatorial_mac() -> None:
    a      = Input(8, 'a_in')
    b      = Input(8, 'b_in')
    accum  = Register(16, 'accumulator')
    
    accum.next <<= a * b + accum

def test_combinatorial_mac():
    reset_working_block()
    combinatorial_mac()
    sim_trace = pyrtl.SimulationTrace()
    sim = pyrtl.Simulation(tracer=sim_trace)
    
    # Test vectors
    inputs = {
        'a_in':    [1, 0, 1, 0, 1, 0, 0, 0],
        'b_in':    [1, 0, 1, 0, 1, 0, 0, 0]
    }
    trace_list=[
        'a_in', 
        'b_in', 
        'accumulator'
    ]
    
    sim.step_multiple(inputs)
    custom_render_trace(sim_trace, trace_list=trace_list, repr_func=lambda x: int(x))

print("Testing pipelined MAC:")
test_combinatorial_mac()

svg = pyrtl.block_to_svg(split_state=False)
display_svg(svg, raw=True)

### Using SimplePipeline


In [8]:
pyrtl.reset_working_block()

class SimpleMAC(SimplePipeline):
    def __init__(self):
        # Create accumulator register outside pipeline stages
        self._accumulator = pyrtl.Register(16, 'accumulator')
        super(SimpleMAC, self).__init__()
        
    def stage0(self):
        # Input stage - receive operands
        self.a = pyrtl.Input(8, 'a_in')
        self.b = pyrtl.Input(8, 'b_in')
        
    def stage1(self):
        # Multiply stage
        self.product = self.a * self.b
        
    def stage2(self):
        # Add stage with accumulation
        self._accumulator.next <<= self.product + self._accumulator

def test_pipeline_mac():
    SimpleMAC()
    sim_trace = pyrtl.SimulationTrace()
    sim = pyrtl.Simulation(tracer=sim_trace)
    
    # Test vectors
    inputs = {
        'a_in':    [1, 0, 1, 0, 1, 0, 0, 0],
        'b_in':    [1, 0, 1, 0, 1, 0, 0, 0]
    }
    trace_list=[
        'a_in', 
        'b_in', 
        'accumulator'
    ]
    
    sim.step_multiple(inputs)
    custom_render_trace(sim_trace, trace_list=trace_list, repr_func=lambda x: int(x))

print("Testing pipelined MAC:")
test_pipeline_mac()

svg = pyrtl.block_to_svg(split_state=False)
display_svg(svg, raw=True)

### Hand built pipelined MAC with reset logic


In [9]:
def mac_8bit(
    a_in: WireVector,
    b_in: WireVector,
    clear: WireVector,
    write_en: WireVector
) -> WireVector:
    """
    8-bit pipelined multiply-accumulate unit
    
    Args:
        a_in: 8-bit multiplicand input
        b_in: 8-bit multiplier input
        clear: 1-bit clear signal to reset accumulator
        write_en: 1-bit write enable for accumulation
        
    Returns:
        out: 16-bit output (max value from accumulating 8x8 products)
    """
    assert len(a_in) == len(b_in) == 8
    assert len(clear) == len(write_en) == 1
    
    # Create pipeline registers for input -> multiply stage
    a_reg = Register(8, 'a_reg')
    b_reg = Register(8, 'b_reg')
    # we_reg = Register(1, 'we_reg')  # Pipeline write enable signal
    
    # Register inputs
    a_reg.next <<= a_in
    b_reg.next <<= b_in
    # we_reg.next <<= write_en
    
    # Multiply stage (8x8=16 bits)
    # product = WireVector(16, 'product')
    # product <<= a_reg * b_reg
    
    # Register product and write enable for multiply -> accumulate stage
    product_reg = Register(16, 'product_reg')
    # we_reg2 = Register(1, 'we_reg2')
    product_reg.next <<= a_reg * b_reg
    # we_reg2.next <<= we_reg
    
    # Accumulator register (16 bits)
    accumulator = Register(16, 'accumulator')
    
    # Accumulate or clear
    with conditional_assignment:
        # with clear:
        #     accumulator.next |= 0
        with write_en:  # Only accumulate when write enable is high
            accumulator.next |= product_reg + accumulator
        with otherwise:
            accumulator.next |= accumulator
            
    return accumulator

def test_mac_8bit():
    reset_working_block()
    
    # Create inputs
    a = Input(8, 'a_in')
    b = Input(8, 'b_in')
    clear = Input(1, 'clear')
    write_en = Input(1, 'write_en')
    
    # Create output
    out = Output(16, 'out')
    
    # Instantiate MAC
    out <<= mac_8bit(a, b, clear, write_en)
    
    # Simulate
    sim_trace = SimulationTrace()
    sim = Simulation(tracer=sim_trace)
    
    # Test vectors:
    # 1. Clear accumulator
    # 2. Accumulate 2*3
    # 3. Accumulate 4*5 (write enabled)
    # 4. Skip accumulation (write disabled)
    # 5. Accumulate 3*3 (write enabled)
    inputs = {
        'a_in':     [0,   2,   4,   5,   3,   0],
        'b_in':     [0,   3,   5,   5,   3,   0],
        'clear':    [1,   0,   0,   0,   0,   0],
        'write_en': [0,   1,   1,   0,   1,   0]
    }
    
    sim.step_multiple(inputs)
    
    # Custom representation for values
    def repr_dec(x): return f"{x:3d}"  # Decimal representation
    def repr_hex(x): return f"0x{x:04x}"  # Hex representation
    
    input_repr_map = {
        'a_in': repr_dec,
        'b_in': repr_dec,
        'clear': repr_dec,
        'write_en': repr_dec,
        'product_reg': repr_hex,
        'accumulator': repr_hex,
        'out': repr_hex
    }
    
    trace_list = list(input_repr_map.keys())
    custom_render_trace(sim_trace, trace_list=trace_list, repr_per_name=input_repr_map)

print("Testing 8-bit MAC:")
test_mac_8bit()

svg = pyrtl.block_to_svg(split_state=False)
display_svg(svg, raw=True)


# with open(output_dir / 'pipeline-mac-sync.v', 'w+') as f:
#     pyrtl.output_to_verilog(f, initialize_registers=True)

# Systolic Array


### Processing Element


**Top inputs**
| i \ j |**top**| 0 | 1 | 2 |
| - | - | - | - | - |
|**left**|
| 0 |
| 1 |
| 2 |

"""
|----------------------------
| {weight_in}
|{data_in}-> | ->{data_out}""
| |
| v
| {weight_out}
|----------------------------
"""


### Test creation of systolic array


### Initialize connections after building structure


## Pyrtl Systolic

### Registers Only

In [ ]:
x = np.array([[1,2,3],[4,5,6],[7,8,9]])
a = np.array([[2,0,0],[0,4,0],[0,0,8]])

data_buffer, weight_buffer = Buffer(x, axis=0), Buffer(x, axis=1)
x, weight_buffer.arr[::-1], data_buffer.arr.T[:,::-1]
a@a

In [53]:
def systolic_reg_array(size: int):
    """
    Generate a square systolic register array of given size
    
    Args:
        size: dimension of square array (size x size)
    """
    # Create input lists for data and weights
    d_in = pyrtl.input_list([f'd_in{i}' for i in range(size)], 8)
    w_in = pyrtl.input_list([f'w_in{i}' for i in range(size)], 8)
    
    # Create 2D arrays of registers for data and weights
    d_reg = [[pyrtl.Register(8, f'd_reg{i}{j}') 
              for j in range(size)] 
              for i in range(size)]
    
    w_reg = [[pyrtl.Register(8, f'w_reg{i}{j}') 
              for j in range(size)] 
              for i in range(size)]
    
    # Create output lists
    d_out = pyrtl.output_list([f'd_out{i}' for i in range(size)], 8)
    w_out = pyrtl.output_list([f'w_out{i}' for i in range(size)], 8)
    
    # Connect data flow (horizontal)
    for i in range(size):
        # First column gets input
        d_reg[i][0].next <<= d_in[i]
        
        # Connect remaining columns
        for j in range(1, size):
            d_reg[i][j].next <<= d_reg[i][j-1]
            
        # Last column connects to output
        d_out[i] <<= d_reg[i][size-1]
    
    # Connect weight flow (vertical)
    for j in range(size):
        # First row gets input
        w_reg[0][j].next <<= w_in[j]
        
        # Connect remaining rows
        for i in range(1, size):
            w_reg[i][j].next <<= w_reg[i-1][j]
            
        # Last row connects to output
        w_out[j] <<= w_reg[size-1][j]

def test_systolic_array():
    reset_working_block()
    
    # Generate a 3x3 array
    systolic_reg_array(3)
    
    # Simulate
    sim_trace = SimulationTrace()
    sim = Simulation(tracer=sim_trace)
    
    # Test vectors
    inputs = {}
    for i in range(3):
        # Data inputs
        inputs[f'd_in{i}'] = [i+1, i+2, i+3, 0, 0]
        # Weight inputs
        inputs[f'w_in{i}'] = [10+i, 20+i, 30+i, 0, 0]
    
    sim.step_multiple(inputs)
    
    # Custom representation
    def repr_dec(x): return f"{x:3d}"
    
    # Create representation map for all signals
    input_repr_map = {}
    trace_list = []
    
    # Add inputs to trace
    for i in range(3):
        name = f'd_in{i}'
        input_repr_map[name] = repr_dec
        trace_list.append(name)
        
    # Add data registers to trace
    for i in range(3):
        for j in range(3):
            name = f'd_reg{i}{j}'
            input_repr_map[name] = repr_dec
            trace_list.append(name)
            
    # Add data outputs to trace
    for i in range(3):
        name = f'd_out{i}'
        input_repr_map[name] = repr_dec
        trace_list.append(name)
        
    # Add weight inputs to trace
    for i in range(3):
        name = f'w_in{i}'
        input_repr_map[name] = repr_dec
        trace_list.append(name)
        
    # Add weight registers to trace
    for i in range(3):
        for j in range(3):
            name = f'w_reg{i}{j}'
            input_repr_map[name] = repr_dec
            trace_list.append(name)
            
    # Add weight outputs to trace
    for i in range(3):
        name = f'w_out{i}'
        input_repr_map[name] = repr_dec
        trace_list.append(name)
    
    custom_render_trace(sim_trace, trace_list=trace_list, repr_per_name=input_repr_map)

print("Testing 3x3 systolic register array:")
test_systolic_array()


In [ ]:
reset_working_block()

def systolic_reg_array_2():
    d_in0, d_in1, w_in0, w_in1 = pyrtl.input_list(['d_in0', 'd_in1', 'w_in0', 'w_in1'], 8)
    
    d_reg00, d_reg01, d_reg10, d_reg11 = pyrtl.register_list(['d_reg00', 'd_reg01', 'd_reg10', 'd_reg11'], 8)
    w_reg00, w_reg01, w_reg10, w_reg11 = pyrtl.register_list(['w_reg00', 'w_reg01', 'w_reg10', 'w_reg11'], 8)
    
    d_out0, d_out1, w_out0, w_out1 = pyrtl.output_list(['d_out0', 'd_out1', 'w_out0', 'w_out1'], 8)
    
    d_reg00.next <<= d_in0
    d_reg10.next <<= d_in1
    w_reg00.next <<= w_in0
    w_reg01.next <<= w_in1

    d_reg01.next <<= d_reg00
    d_reg11.next <<= d_reg10
    d_out0 <<= d_reg01
    d_out1 <<= d_reg11

    w_reg10.next <<= w_reg00
    w_reg11.next <<= w_reg01
    w_out0 <<= w_reg10
    w_out1 <<= w_reg11

systolic_reg_array_2()
svg = pyrtl.block_to_svg(split_state=False)
display_svg(svg, raw=True)

In [ ]:
reset_working_block()

class Buffer:
    def __init__(self, arr: np.ndarray, axis: int) -> None:
        assert axis in [0,1], "Axis must be 0 or 1"
        if axis == 1:
            arr = arr.T
        l = arr.shape[0]
        self.arr = np.array(
            [
                np.pad(arr[i], (l-i-1,i), 'constant', constant_values=(0,0)) 
                for i in range(l)
            ]
        ).T[::-1]

    def __iter__(self):
        for row in self.arr:
            yield row

def systolic_reg_array(size: int, data_in: list[WireVector], weight_in: list[WireVector]):
    registers = [[(Register(8, f"data_{i}_{j}"), Register(8, f"weight_{i}_{j}")) for j in range(size)] for i in range(size)]

    # Initialize the rest
    for row in range(size):
        for col in range(size):
            # print(f"\nUpdating registers in row {row} column {col}")
            if col>0:
                # print(f"Setting {registers[row][col][0].name}.next << {registers[row][col-1][0].name}")
                registers[row][col][0].next <<= registers[row][col-1][0]
            else:
                # print(f"Setting {registers[row][col][0].name}.next << {data_in[row].name}")
                registers[row][col][0].next <<= data_in[row]
            if row>0:
                # print(f"Setting {registers[row][col][1].name}.next << {registers[row-1][col][1].name}")
                registers[row][col][1].next <<= registers[row-1][col][1]
            else:
                # print(f"Setting {registers[row][col][1].name}.next << {weight_in[col].name}")
                registers[row][col][1].next <<= weight_in[col]
    return registers

class SystolicSimulation:
    def __init__(self, data: np.ndarray, weights: np.ndarray, dim=3) -> None:
        reset_working_block()
        self.size = dim
        self.registers = systolic_reg_array(
            dim, 
            data_in=pyrtl.input_list([f'data_in_{i}' for i in range(dim)], 8),
            weight_in=pyrtl.input_list([f'weight_in_{i}' for i in range(dim)], 8)
        )
        self.data = Buffer(data, axis=0)
        self.weights = Buffer(weights, axis=1)
        self.sim = pyrtl.Simulation(tracer=SimulationTrace())
        
        
    def __iter__(self):
        for d,w in zip(self.data, self.weights):
            sim_inputs = {
                **{f'data_in_{i}': v for i, v in enumerate(d)},
                **{f'weight_in_{i}': v for i, v in enumerate(w)}
            }
            print(f"data={d}, weights={w}")
            self.sim.step(sim_inputs)
            yield self.get_current_state()
        for _ in range(self.size):
            sim_inputs = {
                **{f'data_in_{i}': 0 for i in range(self.size)},
                **{f'weight_in_{i}': 0 for i in range(self.size)}
            }
            self.sim.step(sim_inputs)
            yield self.get_current_state()

    def get_current_state(self):
        cur = []
        for row in self.registers:
            cur.append([f"{self.sim.inspect(r[0].name)},{self.sim.inspect(r[1].name)}" for r in row])
        return np.array(cur)

# dim = 2
# data_inputs = pyrtl.input_list([f'data_in_{i}' for i in range(dim)], 8)
# weight_inputs = pyrtl.input_list([f'weight_in_{i}' for i in range(dim)], 8)

# registers = systolic_reg_array(dim, data_inputs, weight_inputs)
# for row in registers:
#     print([(r[0].name,r[1].name) for r in row])

x = np.array([[1,2,3],[4,5,6],[7,8,9]])
a = np.array([[2,0,0],[0,4,0],[0,0,8]])

systolic_sim = SystolicSimulation(x, x, 3)

for state in systolic_sim:
    print(state, '\n')

svg = pyrtl.block_to_svg(split_state=False)
display_svg(svg, raw=True)

In [18]:
systolic_sim.data.arr

### With MAC Units

In [ ]:
def fmac(
    a_in: WireVector,
    b_in: WireVector,
    # clear: WireVector,
    left_en: WireVector,
    up_en: WireVector
) -> WireVector:
    assert len(a_in) == len(b_in) == 8
    assert len(left_en) == len(up_en) == 1 # == len(clear)
    
    # Create pipeline registers for input -> multiply stage
    a_reg = Register(8, 'a_reg')
    b_reg = Register(8, 'b_reg')
    we_reg = Register(1, 'we_reg')  # Pipeline write enable signal
    
    # Register inputs
    a_reg.next <<= a_in
    b_reg.next <<= b_in
    
    # Accumulator register (16 bits)
    accumulator = Register(16, 'accumulator')
    # accumulator.next <<= generalized_fma([(a_reg, b_reg)], accumulator)
    
    # Accumulate or clear
    with conditional_assignment:
        with we_reg:  # Only accumulate when write enable is high
            accumulator.next |= generalized_fma([(a_reg, b_reg)], [accumulator])
        with otherwise:
            accumulator.next |= accumulator
            we_reg.next |= left_en & up_en
            
    return a_reg, b_reg, we_reg, accumulator

def basic_systolic_array(
    matsize: int, 
    left_inputs: List[WireVector], 
    top_inputs: List[WireVector], 
    clear: WireVector, 
    write_en: WireVector
) -> List[WireVector]:
    # Left inputs is a list of inputs to the left side of the systolic array, each wire 8 bits wide

    pe_array = []
    for i in range(matsize): # Iterate over rows
        pe_array.append([])
        for j in range(matsize): # Iterate over columns
            pe_array[i].append(fmac(left_inputs[i], top_inputs[j], clear, write_en))

def MMArray(data_width, matrix_size, data_in, new_weights, weights_in, weights_we):
    '''
    data_in: 256-array of 8-bit activation values from systolic_setup buffer
    new_weights: 256-array of 1-bit control values indicating that new weight should be used
    weights_in: output of weight FIFO (8 x matsize x matsize bit wire)
    weights_we: 1-bit signal to begin writing new weights into the matrix
    '''

    # For signals going to the right, store in a var; for signals going down, keep a list
    # For signals going down, keep a copy of inputs to top row to connect to later
    weights_in_top = [ WireVector(data_width) for i in range(matrix_size) ]  # input weights to top row
    weights_in_last = [x for x in weights_in_top]

    weights_enable_top = [ WireVector(1) for i in range(matrix_size) ]  # weight we to top row
    weights_enable = [x for x in weights_enable_top]

    data_in_left = [ WireVector(data_width) for i in range(matrix_size) ]  # data input to left column (arr[i][0])
    data_in_last = [x for x in data_in_left]

    data_out = [Const(0) for i in range(matrix_size)]  # will hold output from final row

    # Build array of MACs
    for i in range(matrix_size):  # for each row
        din = data_in[i]
        switchin = new_weights[i]
        #probe(switchin, "switch" + str(i))
        for j in range(matrix_size):  # for each column
            left_out, top_out, we_out, accum_out = fmac(data_width, matrix_size, din, data_out[j], switchin, weights_in_last[j], weights_enable[j], weights_tag[j])

            fmac(data_in_last[j], weights_in_last[j], )

            #probe(data_out[j], "MACacc{}_{}".format(i, j))
            #probe(acc_out, "MACout{}_{}".format(i, j))
            #probe(din, "MACdata{}_{}".format(i, j))
            weights_in_last[j] = newweight
            weights_enable[j] = newwe
            weights_tag[j] = newtag
            data_out[j] = acc_out


def test_fmac():
    reset_working_block()
    
    # Create inputs
    a = Input(8, 'a_in')
    b = Input(8, 'b_in')
    clear = Input(1, 'clear')
    write_en = Input(1, 'write_en')
    
    # Create output
    out = Output(16, 'out')
    
    # Instantiate MAC
    out <<= fmac(a, b, clear, write_en)
    
    # Simulate
    sim_trace = SimulationTrace()
    sim = Simulation(tracer=sim_trace)
    
    # Test vectors:
    # 1. Clear accumulator
    # 2. Accumulate 2*3
    # 3. Accumulate 4*5 (write enabled)
    # 4. Skip accumulation (write disabled)
    # 5. Accumulate 3*3 (write enabled)
    inputs = {
        'a_in':     [0,   2,   4,   5,   3,   0],
        'b_in':     [0,   3,   5,   5,   3,   0],
        'clear':    [1,   0,   0,   0,   0,   0],
        'write_en': [0,   1,   1,   0,   1,   0]
    }
    
    sim.step_multiple(inputs)
    
    # Custom representation for values
    def repr_dec(x): return f"{x:3d}"  # Decimal representation
    def repr_hex(x): return f"0x{x:04x}"  # Hex representation
    
    input_repr_map = {
        'a_in': repr_dec,
        'b_in': repr_dec,
        'clear': repr_dec,
        'write_en': repr_dec,
        'accumulator': repr_dec,
        'out': repr_dec
    }
    
    trace_list = list(input_repr_map.keys())
    custom_render_trace(sim_trace, trace_list=trace_list, repr_per_name=input_repr_map)

print("Testing FMA based accumulator:")
test_fmac()

svg = pyrtl.block_to_svg(split_state=False)
display_svg(svg, raw=True)

# FUck this bruh

In [82]:
reset_working_block()

class Buffer:
    def __init__(self, arr: np.ndarray, axis: int) -> None:
        assert axis in [0,1], "Axis must be 0 or 1"
        if axis == 1:
            arr = arr.T
        l = arr.shape[0]
        self.arr = np.array(
            [
                np.pad(arr[i], (l-i-1,i), 'constant', constant_values=(0,0)) 
                for i in range(l)
            ]
        ).T[::-1]

    def __iter__(self):
        for row in self.arr:
            yield row

class SimpleMAC:
    def __init__(self, d_width, acc_size, i, j, reset):
        self.i = i
        self.j = j
        self.data = Register(d_width, f"data_{i}_{j}")
        self.weight = Register(d_width, f"weight_{i}_{j}")
        self.acc = Register(acc_size, f"acc_{i}_{j}")
        self.reset = WireVector(1)
        self.reset <<= reset
        with pyrtl.conditional_assignment:
            with self.reset:
                self.acc.next |= 0
            with pyrtl.otherwise:
                self.acc.next |= self.acc + self.data * self.weight

class SystolicMacArray:
    def __init__(self, data: np.ndarray, weights: np.ndarray, dim=3) -> None:
        reset_working_block()
        self.size = dim
        self.cells = self._build_array(
            dim, 
            data_in=pyrtl.input_list([f'data_in_{i}' for i in range(dim)], 8),
            weight_in=pyrtl.input_list([f'weight_in_{i}' for i in range(dim)], 8)
        )
        self.data = Buffer(data, axis=0)
        self.weights = Buffer(weights, axis=1)
        self.sim = pyrtl.Simulation(tracer=SimulationTrace())
    
    def _build_array(self, size: int, data_in: list[WireVector], weight_in: list[WireVector]):
        # Create the mac units
        reset_wire = Input(1, 'rst')
        cells = [[SimpleMAC(8,16,i,j,reset_wire) for j in range(size)] for i in range(size)]
        # Connect them together
        for row in range(size):
            for col in range(size):
                # Connect data inputs
                if col==0: # First column gets data from the data_in buffer
                    cells[row][col].data.next <<= data_in[row]
                else: # Other columns get data from the "left" neighbor
                    cells[row][col].data.next <<= cells[row][col-1].data
                # Connect weight inputs
                if row==0: # Top row gets weights from the weight_in buffer
                    cells[row][col].weight.next <<= weight_in[col]
                else: # Other rows get weights from the top neighbor
                    cells[row][col].weight.next <<= cells[row-1][col].weight
        return cells
        
    def __iter__(self):
        for d,w in zip(self.data, self.weights):
            sim_inputs = {
                **{f'data_in_{i}': v for i, v in enumerate(d)},
                **{f'weight_in_{i}': v for i, v in enumerate(w)}
            }
            print(f"data={d}, weights={w}")
            self.sim.step(sim_inputs)
            yield self.get_current_state()
        for _ in range(self.size+1):
            sim_inputs = {
                **{f'data_in_{i}': 0 for i in range(self.size)},
                **{f'weight_in_{i}': 0 for i in range(self.size)}
            }
            self.sim.step(sim_inputs)
            yield self.get_current_state()

    def get_current_state(self):
        cur = []
        for row in self.cells:
            cur.append([f"{self.sim.inspect(mac.acc.name)}" for mac in row])
        return np.array(cur)
    
    def matmul(self, data: np.ndarray, weights: np.ndarray, v=True):
        z = np.zeros((self.size, self.size+1))
        data_buffer = np.concat([Buffer(data, axis=0).arr.T, z], axis=1)
        weight_buffer = np.concat([Buffer(weights, axis=1).arr.T, z], axis=1)
        sim_inputs = {
            **{f'data_in_{i}': v for i, v in enumerate(data_buffer)},
            **{f'weight_in_{i}': v for i, v in enumerate(weight_buffer)}
        }
        print(sim_inputs)
        sim = Simulation()
        sim.step_multiple(sim_inputs, nsteps=4)
        result = self.get_current_state()
        print("Systolic Result:", result)
        print("True Result:", data@weights)
        return result


x = np.array([[1,2,3],[4,5,6],[7,8,9]])
a = np.array([[2,0,0],[0,4,0],[0,0,8]])
i = np.array([[0,0,1],[0,0,0],[1,0,0]])

systolic_sim = SystolicMacArray(x, a, 3)

# for state in systolic_sim:
#     print(state, '\n')
systolic_sim.matmul(x,a)

svg = pyrtl.block_to_svg(split_state=False)
display_svg(svg, raw=True)

In [67]:
np.concat([np.zeros((3,1)), np.ones((3,1))])

# Multiply and Accumulate


### Setup
